In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')


# 1. YÜKLEME VE SÜTUN TEMİZLİĞİ
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')

# Senin en iyi 1.203'lük dosyanı alıyoruz
best_submission_file = 'Takim_141_version_7.csv'
pseudo_labels = pd.read_csv(best_submission_file)

train.columns = train.columns.str.lower().str.replace(' ', '_').str.strip()
test.columns = test.columns.str.lower().str.replace(' ', '_').str.strip()
target_col = 'bilissel_performans_skoru'

test_pseudo = test.copy()
test_pseudo[target_col] = pseudo_labels[target_col]

combined_train = pd.concat([train, test_pseudo], axis=0).reset_index(drop=True)

# 3. ÖZELLİK MÜHENDİSLİĞİ (Güvenli olanlar)
def create_safe_features(df):
    df = df.copy()
    if 'rem_yuzdesi' in df.columns and 'derin_uyku_yuzdesi' in df.columns:
        df['toplam_kaliteli_uyku'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    if 'uyku_oncesi_kafein_mg' in df.columns and 'uyku_oncesi_ekran_suresi_dk' in df.columns:
        df['uyku_bozucu_faktorler'] = (df['uyku_oncesi_kafein_mg']/100) + (df['uyku_oncesi_ekran_suresi_dk']/30)
    if 'gunluk_calisma_saati' in df.columns and 'stres_skoru' in df.columns:
        df['tukenmislik_riski'] = df['gunluk_calisma_saati'] * df['stres_skoru']
    return df

combined_train = create_safe_features(combined_train)
test = create_safe_features(test)
train = create_safe_features(train) # Added: Apply feature engineering to original train DataFrame

y_combined = combined_train[target_col]
X_combined = combined_train.drop(['id', target_col], axis=1)
X_test = test.drop(['id'], axis=1)

# 4. LIGHTGBM İÇİN LABEL ENCODING (LightGBM string sevmez, sayı ister)
num_cols = X_combined.select_dtypes(include=[np.number]).columns
for col in num_cols:
    med_val = train[col].median()
    X_combined[col] = X_combined[col].fillna(med_val)
    X_test[col] = X_test[col].fillna(med_val)

cat_cols = X_combined.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    X_combined[col] = X_combined[col].fillna('Missing').astype(str)
    X_test[col] = X_test[col].fillna('Missing').astype(str)

    le = LabelEncoder()
    # Tüm birleşik veri üzerinden fit ediyoruz
    X_combined[col] = le.fit_transform(X_combined[col])
    # Test verisini güvenli transform ediyoruz
    X_test[col] = X_test[col].apply(lambda x: x if x in le.classes_ else 'Missing')
    X_test[col] = le.transform(X_test[col])

# 5. MULTI-SEED LIGHTGBM EĞİTİMİ
SEEDS = [42, 123, 777, 2024, 9999]
lgb_predictions = np.zeros(len(X_test))

print("LightGBM Modelleri Eğitiliyor...")

for i, seed in enumerate(SEEDS):
    print(f" -> LGB Model {i+1}/5 (Seed: {seed})...")

    model = lgb.LGBMRegressor(
        n_estimators=1500,
        learning_rate=0.015,
        max_depth=7,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        verbose=-1
    )

    model.fit(X_combined, y_combined)
    lgb_predictions += model.predict(X_test) / len(SEEDS)

# 6. ALTIN VURUŞ: CATBOOST VE LIGHTGBM GÜÇLERİNİ BİRLEŞTİR (BLENDING)
print("\n" + "="*60)
print("CATBOOST + LIGHTGBM")
print("="*60)

# CatBoost tahminlerini (1.203) çekiyoruz
catboost_preds = pseudo_labels[target_col].values

# %60 CatBoost, %40 LightGBM (CatBoost bu verisetinde biraz daha güçlü)
final_blended_scores = (catboost_preds * 0.60) + (lgb_predictions * 0.40)
final_blended_scores = np.clip(final_blended_scores, 0, 10)

submission = pd.DataFrame({
    'id': test['id'],
    'bilissel_performans_skoru': final_blended_scores
})

submission.to_csv('Takim_141_version_8.csv', index=False)
print("DOSYA HAZIR: Takim_141_version_8.csv")

LightGBM Modelleri Eğitiliyor...
 -> LGB Model 1/5 (Seed: 42)...
 -> LGB Model 2/5 (Seed: 123)...
 -> LGB Model 3/5 (Seed: 777)...
 -> LGB Model 4/5 (Seed: 2024)...
 -> LGB Model 5/5 (Seed: 9999)...

CATBOOST + LIGHTGBM
DOSYA HAZIR: Takim_141_version_8.csv
